In [2]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
api_key = os.getenv("DART_API_KEY")

# DART CORPCODE.xml 다운로드 URL
url = "https://opendart.fss.or.kr/api/corpCode.xml"
params = {"crtfc_key": api_key}

print("CORPCODE.xml 다운로드 중...")
response = requests.get(url, params=params)
print(f"응답 코드: {response.status_code}")
print(f"파일 크기: {len(response.content):,} bytes ({len(response.content) / 1024 / 1024:.2f} MB)")
print(f"Content-Type: {response.headers.get('Content-Type')}")

CORPCODE.xml 다운로드 중...
응답 코드: 200
파일 크기: 3,579,368 bytes (3.41 MB)
Content-Type: application/x-msdownload;charset=UTF-8


In [3]:
import zipfile
import io
import xml.etree.ElementTree as ET

# response.content는 ZIP 바이너리
# 디스크에 저장하지 않고 메모리에서 바로 압축 해제
zf = zipfile.ZipFile(io.BytesIO(response.content))

# ZIP 안에 있는 파일 목록 확인
print("ZIP 내부 파일:", zf.namelist())

# CORPCODE.xml 읽기
xml_data = zf.read("CORPCODE.xml")
print(f"XML 크기: {len(xml_data):,} bytes ({len(xml_data) / 1024 / 1024:.2f} MB)")

# XML 첫 500자 미리보기
print("\n=== XML 미리보기 ===")
print(xml_data[:500].decode("utf-8"))

ZIP 내부 파일: ['CORPCODE.xml']
XML 크기: 29,912,902 bytes (28.53 MB)

=== XML 미리보기 ===
<?xml version="1.0" encoding="UTF-8"?>
<result>
    <list>
        <corp_code>00434003</corp_code>
        <corp_name>다코</corp_name>
        <corp_eng_name>Daco corporation</corp_eng_name>
        <stock_code> </stock_code>
        <modify_date>20170630</modify_date>
    </list>
    <list>
        <corp_code>00430964</corp_code>
        <corp_name>굿앤엘에스</corp_name>
        <corp_eng_name>Good &amp; LS Co.,Ltd.</corp_eng_name>
        <stock_code> </stock_code>
        <modify_date>


In [5]:
import pandas as pd

# XML 파싱
root = ET.fromstring(xml_data)

# 각 <list> 요소에서 필요한 필드 추출
rows = []
for child in root.iter("list"):
    rows.append({
        "corp_code": child.findtext("corp_code"),
        "corp_name": child.findtext("corp_name"),
        "stock_code": child.findtext("stock_code"),
        "modify_date": child.findtext("modify_date"),
    })

# DataFrame으로 변환
df_all = pd.DataFrame(rows)

print(f"전체 기업 수: {len(df_all):,}개")
print(f"\n컬럼: {list(df_all.columns)}")
print(f"\n=== 상위 5개 ===")
print(df_all.head())

전체 기업 수: 118,145개

컬럼: ['corp_code', 'corp_name', 'stock_code', 'modify_date']

=== 상위 5개 ===
  corp_code          corp_name stock_code modify_date
0  00434003                 다코               20170630
1  00430964              굿앤엘에스               20170630
2  00388953  크레디피아제이십오차유동화전문회사               20170630
3  00179984             연방건설산업               20170630
4  00420143     브룩스피알아이오토메이션잉크               20170630


In [6]:
# 종목코드가 공백이 아닌 것만 = 상장사
listed = df_all[df_all["stock_code"].str.strip() != ""].reset_index(drop=True)

print(f"상장사 수: {len(listed):,}개")
print(f"\n=== 상장사 상위 10개 ===")
print(listed.head(10))

# 우리가 잘 아는 삼성전자 확인
samsung = listed[listed["corp_name"] == "삼성전자"]
print(f"\n=== 삼성전자 확인 ===")
print(samsung)

상장사 수: 3,967개

=== 상장사 상위 10개 ===
  corp_code corp_name stock_code modify_date
0  00260985      한빛네트     036720    20170630
1  00264529      엔플렉스     040130    20170630
2  00358545    동서정보기술     055000    20170630
3  00231567     애드모바일     032600    20170630
4  00359614       리더컴     056140    20170630
5  00153551    허메스홀딩스     012400    20170630
6  00344746      유티엑스     045880    20170630
7  00261188     글로포스트     037830    20170630
8  00268020      쏠라엔텍     030390    20170630
9  00269287        보홍     041320    20170630

=== 삼성전자 확인 ===
     corp_code corp_name stock_code modify_date
3374  00126380      삼성전자     005930    20251201


In [8]:
# Plan B: pykrx 완전 우회
# 시장 구분 없이 매핑 테이블 완성

print("=" * 60)
print("Plan B: pykrx 우회 모드")
print("=" * 60)

# market 컬럼만 추가 (값은 UNKNOWN)
listed["market"] = "UNKNOWN"

print(f"\n✅ 총 상장사: {len(listed):,}개")
print(f"\n=== 상위 10개 미리보기 ===")
print(listed.head(10))

Plan B: pykrx 우회 모드

✅ 총 상장사: 3,967개

=== 상위 10개 미리보기 ===
  corp_code corp_name stock_code modify_date   market
0  00260985      한빛네트     036720    20170630  UNKNOWN
1  00264529      엔플렉스     040130    20170630  UNKNOWN
2  00358545    동서정보기술     055000    20170630  UNKNOWN
3  00231567     애드모바일     032600    20170630  UNKNOWN
4  00359614       리더컴     056140    20170630  UNKNOWN
5  00153551    허메스홀딩스     012400    20170630  UNKNOWN
6  00344746      유티엑스     045880    20170630  UNKNOWN
7  00261188     글로포스트     037830    20170630  UNKNOWN
8  00268020      쏠라엔텍     030390    20170630  UNKNOWN
9  00269287        보홍     041320    20170630  UNKNOWN


In [9]:
import os

# CSV로 저장
output_path = "../data/corp_map.csv"
listed.to_csv(output_path, index=False, encoding="utf-8-sig")

file_size = os.path.getsize(output_path)
print(f"✅ 저장 완료: {output_path}")
print(f"파일 크기: {file_size:,} bytes ({file_size / 1024:.1f} KB)")
print(f"행 수: {len(listed):,}개")

✅ 저장 완료: ../data/corp_map.csv
파일 크기: 201,656 bytes (196.9 KB)
행 수: 3,967개


In [10]:
import pandas as pd

# 저장한 CSV를 다시 읽어서 검증
df_check = pd.read_csv(
    "../data/corp_map.csv",
    dtype={"corp_code": str, "stock_code": str}
)

print(f"읽어온 종목 수: {len(df_check):,}개\n")

print("=== 5종목 샘플 확인 ===")
samples = ["삼성전자", "SK하이닉스", "카카오", "셀트리온", "KB금융"]
for name in samples:
    row = df_check[df_check["corp_name"] == name]
    if not row.empty:
        r = row.iloc[0]
        print(f"  ✅ {name}: corp_code={r['corp_code']}, stock_code={r['stock_code']}")
    else:
        print(f"  ⚠️ {name} 못 찾음")

읽어온 종목 수: 3,967개

=== 5종목 샘플 확인 ===
  ✅ 삼성전자: corp_code=00126380, stock_code=005930
  ✅ SK하이닉스: corp_code=00164779, stock_code=000660
  ✅ 카카오: corp_code=00258801, stock_code=035720
  ✅ 셀트리온: corp_code=00413046, stock_code=068270
  ✅ KB금융: corp_code=00688996, stock_code=105560
